In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
import torchvision
import torchvision.transforms as transforms
import os
import argparse
from pathlib import Path
import re
import random
import math
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Model and Loss

In [3]:
"""SAITS source-aligned, self-contained PyTorch architecture for (B, 1080, 2).

Architecture adapted from Du, Cote & Liu, SAITS, Expert Systems with
Applications 219 (2023), 119619. Official MIT-licensed PyTorch code:
https://github.com/WenjieDu/SAITS/blob/main/modeling/saits.py
https://github.com/WenjieDu/SAITS/blob/main/modeling/layers.py

Author-implementation architecture retained:
  * two blocks of diagonally-masked (optional) self-attention;
  * separate Q/K/V projections, pre-layernorm attention and FFN;
  * group-based weight-sharing options (inner_group/between_group);
  * first-stage linear reconstruction, second-stage beta-ReLU-gamma;
  * sigmoid gate over [observed mask, last second-block attention weights];
  * correct final observed-value complementation and original ORT/MIT MAE.

Interface adapted to your masked multivariate time series, x_in/m_hat tensors.
Numerical/engineering adaptations: stable nonfinite masking in the objective;
PyTorch scaled-dot-product attention for intermediate layers by default
(mathematically the same softmax attention, not an architectural change).
Using 'fast_attention=False' computes explicit attention at every layer.
This is a source-aligned port, not a drop-in load of official checkpoints.

Paper-compatible loss formula: lam_ort * mean(MAE(tilde1), MAE(tilde2),
MAE(tilde3)) on observed entries + lam_mit * MAE(tilde3) on held-out entries.
Select lam_ort / lam_mit in the training script.
"""
import math
import torch
from torch import nn
from torch.nn import functional as F


class _SAITSPositionEncoding(nn.Module):
    """Fixed sinusoidal encoding from the authors' implementation."""
    def __init__(self, d_model, seq_len):
        super().__init__()
        pos = torch.arange(seq_len, dtype=torch.float32).unsqueeze(1)
        j = torch.arange(d_model, dtype=torch.float32).unsqueeze(0)
        phase = pos / torch.pow(10000.0, 2 * torch.floor(j / 2) / d_model)
        pe = torch.empty(seq_len, d_model)
        pe[:, 0::2] = torch.sin(phase[:, 0::2])
        pe[:, 1::2] = torch.cos(phase[:, 1::2])
        self.register_buffer('pos_table', pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        if x.size(1) > self.pos_table.size(1):
            raise ValueError('Sequence length exceeds positional encoding length')
        return x + self.pos_table[:, :x.size(1)].to(dtype=x.dtype)


class _SAITSMultiHeadAttention(nn.Module):
    """Same Q/K/V parameterization and attention math as modeling/layers.py."""
    def __init__(self, d_model, n_heads, d_k, d_v, attention_dropout=0.0,
                 fast_attention=True):
        super().__init__()
        self.n_heads, self.d_k, self.d_v = n_heads, d_k, d_v
        self.attention_dropout = attention_dropout
        self.fast_attention = fast_attention
        self.w_qs = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.w_ks = nn.Linear(d_model, n_heads * d_k, bias=False)
        self.w_vs = nn.Linear(d_model, n_heads * d_v, bias=False)
        self.fc = nn.Linear(n_heads * d_v, d_model, bias=False)
        self.attention_drop = nn.Dropout(attention_dropout)

    def forward(self, x, diagonal_attention_mask, return_weights=False):
        B, T, _ = x.shape
        H = self.n_heads
        q = self.w_qs(x).reshape(B, T, H, self.d_k).transpose(1, 2)
        k = self.w_ks(x).reshape(B, T, H, self.d_k).transpose(1, 2)
        v = self.w_vs(x).reshape(B, T, H, self.d_v).transpose(1, 2)

        if self.fast_attention and not return_weights:
            allowed = None
            if diagonal_attention_mask:
                allowed = ~torch.eye(T, dtype=torch.bool, device=x.device)
            z = F.scaled_dot_product_attention(
                q, k, v, attn_mask=allowed,
                dropout_p=self.attention_dropout if self.training else 0.0,
                is_causal=False,
            )
            weights = None
        else:
            logits = (q / math.sqrt(self.d_k)) @ k.transpose(-2, -1)
            if diagonal_attention_mask:
                diagonal = torch.eye(T, dtype=torch.bool, device=x.device)
                logits = logits.masked_fill(
                    diagonal[None, None],
                    -1e4 if logits.dtype == torch.float16 else -1e9,
                )
            weights = self.attention_drop(torch.softmax(logits, dim=-1))
            z = weights @ v

        z = z.transpose(1, 2).contiguous().reshape(B, T, H * self.d_v)
        return self.fc(z), weights


class _SAITSFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model, eps=1e-6)
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        return x + self.drop(self.fc2(F.relu(self.fc1(self.norm(x)))))


class _SAITSEncoderLayer(nn.Module):
    """Official pre-LN residual attention + pre-LN residual feedforward."""
    def __init__(self, d_model, d_ff, n_heads, d_k, d_v, dropout,
                 attn_dropout, diagonal_attention_mask, fast_attention):
        super().__init__()
        self.diagonal_attention_mask = diagonal_attention_mask
        self.norm = nn.LayerNorm(d_model)  # official default eps=1e-5
        self.attn = _SAITSMultiHeadAttention(
            d_model, n_heads, d_k, d_v, attn_dropout, fast_attention
        )
        self.drop = nn.Dropout(dropout)
        self.ffn = _SAITSFeedForward(d_model, d_ff, dropout)

    def forward(self, x, return_attention=False):
        y, weights = self.attn(
            self.norm(x), self.diagonal_attention_mask,
            return_weights=return_attention,
        )
        return self.ffn(x + self.drop(y)), weights


class SAITSImputerStandard(nn.Module):
    """Original SAITS architecture with x_in/m_hat call signature.

    Defaults retain your two-layer/128-dim comparison capacity. To reproduce
    the authors' PhysioNet2012 *hyperparameters*, use n_groups=5,
    n_group_inner_layers=1, param_sharing_strategy='inner_group',
    d_model=256, d_ff=512, n_heads=8, d_k=32, d_v=32, dropout=0.
    That hyperparameter choice is NOT guaranteed best for 1080-step data.

    forward(x_in,m_hat) returns (tilde1, tilde2, tilde3, x_c, attn2_mean).
    """
    def __init__(
        self,
        seq_len=1080, d_in=2, d_model=128, n_heads=8, d_ff=256,
        n_groups=2, n_group_inner_layers=1,
        param_sharing_strategy='inner_group', d_k=None, d_v=None,
        dropout=0.1, attn_dropout=0.0, input_with_mask=True,
        diagonal_attention_mask=True, fast_attention=True,
    ):
        super().__init__()
        if n_groups < 1 or n_group_inner_layers < 1:
            raise ValueError('Both n_groups and n_group_inner_layers must be positive')
        if param_sharing_strategy not in ('inner_group', 'between_group'):
            raise ValueError("param_sharing_strategy must be 'inner_group' or 'between_group'")
        if seq_len < 2 and diagonal_attention_mask:
            raise ValueError('Diagonal masked attention requires at least 2 time steps')
        if d_k is None:
            if d_model % n_heads:
                raise ValueError('d_model must be divisible by n_heads when d_k unspecified')
            d_k = d_model // n_heads
        if d_v is None:
            d_v = d_k
        self.seq_len, self.d_in = int(seq_len), int(d_in)
        self.n_groups = int(n_groups)
        self.n_group_inner_layers = int(n_group_inner_layers)
        self.param_sharing_strategy = param_sharing_strategy
        self.input_with_mask = bool(input_with_mask)
        dim_input = 2 * d_in if input_with_mask else d_in
        layer_kwargs = dict(
            d_model=d_model, d_ff=d_ff, n_heads=n_heads, d_k=d_k, d_v=d_v,
            dropout=dropout, attn_dropout=attn_dropout,
            diagonal_attention_mask=diagonal_attention_mask,
            fast_attention=fast_attention,
        )
        count = n_group_inner_layers if param_sharing_strategy == 'between_group' else n_groups
        self.first_block = nn.ModuleList([_SAITSEncoderLayer(**layer_kwargs) for _ in range(count)])
        self.second_block = nn.ModuleList([_SAITSEncoderLayer(**layer_kwargs) for _ in range(count)])
        self.dropout = nn.Dropout(dropout)
        self.position_enc = _SAITSPositionEncoding(d_model, seq_len)
        self.embedding_1 = nn.Linear(dim_input, d_model)
        self.embedding_2 = nn.Linear(dim_input, d_model)
        self.reduce_dim_z = nn.Linear(d_model, d_in)
        self.reduce_dim_beta = nn.Linear(d_model, d_in)
        self.reduce_dim_gamma = nn.Linear(d_in, d_in)
        self.weight_combine = nn.Linear(d_in + seq_len, d_in)

    def _encode_block(self, x, layers, return_last_attention):
        # Both loops reproduce parameter-sharing semantics in official code.
        last_weights = None
        if self.param_sharing_strategy == 'between_group':
            for grp in range(self.n_groups):
                for j, layer in enumerate(layers):
                    last = (grp == self.n_groups - 1 and j == len(layers) - 1)
                    x, weights = layer(x, return_attention=return_last_attention and last)
                    if return_last_attention and last:
                        last_weights = weights
        else:
            for i, layer in enumerate(layers):
                for inner in range(self.n_group_inner_layers):
                    last = (i == len(layers) - 1 and inner == self.n_group_inner_layers - 1)
                    x, weights = layer(x, return_attention=return_last_attention and last)
                    if return_last_attention and last:
                        last_weights = weights
        return x, last_weights

    def forward(self, x_in, m_hat):
        if x_in.ndim != 3 or x_in.shape[1:] != (self.seq_len, self.d_in):
            raise ValueError(f'Expected (B,{self.seq_len},{self.d_in}), got {tuple(x_in.shape)}')
        if m_hat.shape != x_in.shape:
            raise ValueError('m_hat and x_in must have same shape')
        mask = ((m_hat > 0.5) & torch.isfinite(x_in)).to(x_in.dtype)
        observed_x = torch.where(mask.bool(), x_in, torch.zeros_like(x_in))
        first_input = torch.cat((observed_x, mask), dim=-1) if self.input_with_mask else observed_x
        z = self.dropout(self.position_enc(self.embedding_1(first_input)))
        z, _ = self._encode_block(z, self.first_block, return_last_attention=False)
        x_tilde_1 = self.reduce_dim_z(z)
        x_prime = torch.where(mask.bool(), observed_x, x_tilde_1)

        second_input = torch.cat((x_prime, mask), dim=-1) if self.input_with_mask else x_prime
        y = self.position_enc(self.embedding_2(second_input))
        y, attn = self._encode_block(y, self.second_block, return_last_attention=True)
        x_tilde_2 = self.reduce_dim_gamma(F.relu(self.reduce_dim_beta(y)))
        attn_mean = attn.mean(dim=1)  # (B, T, T)
        eta = torch.sigmoid(self.weight_combine(torch.cat((mask, attn_mean), dim=-1)))
        x_tilde_3 = (1.0 - eta) * x_tilde_2 + eta * x_tilde_1
        x_c = torch.where(mask.bool(), observed_x, x_tilde_3)
        return x_tilde_1, x_tilde_2, x_tilde_3, x_c, attn_mean


def masked_mae(estimation, target, mask, eps=1e-9):
    """Original MAE ratio with robust exclusion of nonfinite target values."""
    if estimation.shape != target.shape or estimation.shape != mask.shape:
        raise ValueError('estimation, target and mask must have identical shapes')
    valid = (mask > 0.5) & torch.isfinite(target)
    target_safe = torch.where(valid, target, torch.zeros_like(target))
    errors = (estimation - target_safe).abs()
    errors = torch.where(valid, errors, torch.zeros_like(errors))
    return errors.sum() / valid.sum().clamp_min(1).to(errors.dtype)


def saits_losses_standard(x_tilde_1, x_tilde_2, x_tilde_3, x_c,
                          x_true, m_hat, i_mask, lam_ort=1.0, lam_mit=1.0):
    """Paper's ORT/MIT objective; no test ground truth is passed to the model.

    ORT: average MAE across all 3 reconstructions at observed input positions.
    MIT: MAE of final reconstruction x_tilde_3 at artificially hidden positions.
    M_hat is 1 for observed in x_in; I_mask is 1 for artificially withheld GT.
    Ground truth at genuine unavailable entries may be NaN.
    """
    if torch.any((m_hat > 0.5) & (i_mask > 0.5)):
        raise ValueError('m_hat and i_mask must be disjoint')
    observed = (m_hat > 0.5) & (i_mask <= 0.5)
    heldout = (i_mask > 0.5) & (m_hat <= 0.5)
    ort_1 = masked_mae(x_tilde_1, x_true, observed)
    ort_2 = masked_mae(x_tilde_2, x_true, observed)
    ort_3 = masked_mae(x_tilde_3, x_true, observed)
    ort = (ort_1 + ort_2 + ort_3) / 3.0
    mit = masked_mae(x_tilde_3, x_true, heldout)
    total = lam_ort * ort + lam_mit * mit
    return total, ort, mit


### Model training

In [4]:
NPZ_PATH = '/content/drive/MyDrive/aim3/saits_masks_1800x2_reintroduced_exact_gaps_capped360_with_bio_numeric.npz'
OUT_DIR = '/content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64'

In [5]:
"""SAITS source-aligned architecture: five-fold patient-level CV on (N,1080,2).

Run AFTER executing saits_official_architecture_1080x2.py model definition in
THE SAME Colab namespace (for files use `%run -i path/to/this_script.py`).
This file does NOT redefine the SAITS model or its MAE ORT/MIT loss.

Expected NPZ:
  X_in: masked input (N,T,2), M_hat: input observed indicator
  X: ground truth (NaN allowed at naturally unavailable values)
  I: artificially withheld indicator (disjoint from M_hat)
  segment_ids: patient IDs

Training: authors' SAITS loss = lambda_ORT * mean(ORT1,2,3 MAE)
                            + lambda_MIT * MIT MAE.
Validation: checkpoint selected by MAE at artificially withheld validation points.
Testing: MAE/RMSE/MAPE on held-out patients, stratified by exact gap lengths.

Study adaptation: normalization uses only observed TRAIN-patient values, applied
with the same constants to train/validation/test; final metrics in raw units.
Model architecture is defined separately in saits_official_architecture_1080x2.py.
"""

import os
import random
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


MODEL_NAME = 'SAITS_OfficialArchitecture'
SEED = 1337
N_SPLITS = 5
VAL_WITHIN_TRAIN_P = 0.10
EPOCHS = 300
EARLY_STOPPING_PATIENCE = 15
BATCH_SIZE = 64                 
GRAD_ACCUM_STEPS = 4       
EVAL_BATCH_SIZE = 4
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
SEQ_LEN, D_IN = 1080, 2

LAM_ORT, LAM_MIT = 1.0, 1.0
NORMALIZE_WITH_TRAIN_STATS = True
USE_AMP = True
GRAD_CLIP_NORM = 1.0
MAPE_EPS = 1e-8

MODEL_KWARGS = dict(
    seq_len=SEQ_LEN, d_in=D_IN, d_model=256, n_heads=8,
    d_k=32, d_v=32, d_ff=512,
    n_groups=5, n_group_inner_layers=1,
    param_sharing_strategy='inner_group',
    dropout=0.0, attn_dropout=0.0,
    input_with_mask=True, diagonal_attention_mask=True,
    fast_attention=True,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
AMP_ENABLED = USE_AMP and DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if AMP_ENABLED and torch.cuda.is_bf16_supported() else torch.float16
REGION_ORDER = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min']


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def autocast_ctx():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if AMP_ENABLED else nullcontext()


def make_scaler():
    enabled = AMP_ENABLED and AMP_DTYPE == torch.float16
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)


def compute_train_stats(x_input, m_obs, indices):
    """Stats from actually observed TRAIN patient inputs only; no target leakage."""
    observed = (m_obs[indices] > 0.5) & np.isfinite(x_input[indices])
    means = np.zeros(D_IN, dtype=np.float32)
    stds = np.ones(D_IN, dtype=np.float32)
    if NORMALIZE_WITH_TRAIN_STATS:
        for d in range(D_IN):
            vals = np.asarray(x_input[indices, :, d])[observed[:, :, d]]
            if vals.size == 0:
                raise ValueError(f'No observed training data for channel {d}')
            means[d] = float(np.mean(vals, dtype=np.float64))
            stds[d] = max(float(np.std(vals, dtype=np.float64)), 1e-6)
    return means, stds


class TSImputeDataset(Dataset):
    def __init__(self, x_in, obs, truth, holdout, original_indices, means, stds):
        self.indices = torch.from_numpy(np.asarray(original_indices, dtype=np.int64))
        obs_arr = np.asarray(obs) > 0.5
        data_arr = np.asarray(x_in, dtype=np.float32)
        truth_arr = np.asarray(truth, dtype=np.float32)
        stats_mu = np.asarray(means, dtype=np.float32).reshape(1, 1, -1)
        stats_std = np.asarray(stds, dtype=np.float32).reshape(1, 1, -1)
        # Zero-fill after standardizing; missing values never become model inputs.
        xin = np.where(obs_arr, (data_arr - stats_mu) / stats_std, 0.0)
        # Keep NaNs for naturally unobserved GT; the MAE loss masks them safely.
        y = np.where(np.isfinite(truth_arr), (truth_arr - stats_mu) / stats_std, np.nan)
        self.x_in = torch.from_numpy(np.nan_to_num(xin, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32))
        self.m_hat = torch.from_numpy(obs_arr.astype(np.float32))
        self.x_true = torch.from_numpy(y.astype(np.float32))
        self.i_mask = torch.from_numpy((np.asarray(holdout) > 0.5).astype(np.float32))
        assert self.x_in.shape == self.m_hat.shape == self.x_true.shape == self.i_mask.shape

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.x_in[i], self.m_hat[i], self.x_true[i], self.i_mask[i], self.indices[i]


def load_loader(idx, shuffle, x_in, m_hat, x_true, i_mask, means, stds, *, evaluation=False):
    ds = TSImputeDataset(x_in[idx], m_hat[idx], x_true[idx], i_mask[idx], idx, means, stds)
    return DataLoader(ds, batch_size=EVAL_BATCH_SIZE if evaluation else BATCH_SIZE,
                      shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


def run_loss_epoch(model, loader, optimizer=None, scaler=None):
    """Returns full-dataset ORT, MIT, combined losses, weighted by valid points."""
    training = optimizer is not None
    model.train(training)
    ort_sum = mit_sum = ort_n = mit_n = 0.0
    if training:
        optimizer.zero_grad(set_to_none=True)

    with torch.set_grad_enabled(training):
        for batch_idx, (xb, mb, yb, ib, _) in enumerate(loader):
            xb = xb.to(DEVICE, non_blocking=True)
            mb = mb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            ib = ib.to(DEVICE, non_blocking=True)
            n_ort = int(((mb > 0.5) & torch.isfinite(yb)).sum().item())
            n_mit = int(((ib > 0.5) & (mb <= 0.5) & torch.isfinite(yb)).sum().item())
            if n_ort == 0 and n_mit == 0:
                continue
            with autocast_ctx():
                t1, t2, t3, xc, _ = model(xb, mb)
                loss, ort, mit = saits_losses_standard(
                    t1, t2, t3, xc, yb, mb, ib,
                    lam_ort=LAM_ORT, lam_mit=LAM_MIT,
                )
            if not torch.isfinite(loss):
                raise FloatingPointError(f'Nonfinite SAITS loss at batch {batch_idx}')

            if training:
                # Correct divisor for the final (possibly short) accumulation group.
                group_start = (batch_idx // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
                group_size = min(GRAD_ACCUM_STEPS, len(loader) - group_start)
                scaler.scale(loss / group_size).backward()
                if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or batch_idx + 1 == len(loader):
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
            ort_sum += float(ort.detach()) * n_ort
            mit_sum += float(mit.detach()) * n_mit
            ort_n += n_ort
            mit_n += n_mit

    ort_avg = ort_sum / ort_n if ort_n else float('nan')
    mit_avg = mit_sum / mit_n if mit_n else float('nan')
    total = LAM_ORT * (ort_avg if np.isfinite(ort_avg) else 0.) + LAM_MIT * (mit_avg if np.isfinite(mit_avg) else 0.)
    return dict(total=float(total), ort=float(ort_avg), mit=float(mit_avg), n_ort=int(ort_n), n_mit=int(mit_n))


def gap_lengths(mask):
    m = np.asarray(mask) > 0.5
    changes = np.flatnonzero(np.diff(np.r_[False, m, False].astype(np.int8)))
    lengths = np.zeros(len(m), dtype=np.int32)
    for start, end in zip(changes[::2], changes[1::2]):
        lengths[start:end] = end - start
    return lengths


def gap_region(g):
    # dt=5 seconds; the gap with 12 samples lasts exactly 1 minute.
    if 1 <= g < 12: return '<1 min'
    if 12 <= g < 120: return '[1,10) min'
    if 120 <= g < 240: return '[10,20) min'
    if 240 <= g <= 360: return '[20,30] min'
    return None


@torch.inference_mode()
def collect_predictions(model, loader, fold, split, patient_ids, filenames, target_names, means, stds):
    model.eval()
    rows = []
    mu = np.asarray(means, dtype=np.float32).reshape(1, 1, D_IN)
    sd = np.asarray(stds, dtype=np.float32).reshape(1, 1, D_IN)
    for xb, mb, truth, holdout, indices in loader:
        with autocast_ctx():
            _, _, _, completed, _ = model(xb.to(DEVICE, non_blocking=True), mb.to(DEVICE, non_blocking=True))
        preds = completed.float().cpu().numpy() * sd + mu
        actual = truth.numpy() * sd + mu
        holdout = holdout.numpy()
        for b, idx_tensor in enumerate(indices):
            idx = int(idx_tensor)
            for d in range(D_IN):
                lengths = gap_lengths(holdout[b, :, d])
                for t in np.flatnonzero(holdout[b, :, d] > 0.5):
                    yt, yp = float(actual[b, t, d]), float(preds[b, t, d])
                    if not (np.isfinite(yt) and np.isfinite(yp)):
                        continue
                    g = int(lengths[t])
                    rows.append(dict(
                        fold=fold, split=split, model=MODEL_NAME,
                        segment_idx=idx, patient_id=str(patient_ids[idx]), filename=str(filenames[idx]),
                        time_idx=int(t), target=str(target_names[d]), y_true=yt, y_pred=yp,
                        abs_err=abs(yt - yp), gap_len=g, gap_region=gap_region(g),
                    ))
    return pd.DataFrame(rows)


def add_metrics(df, keys):
    if df.empty:
        return pd.DataFrame(columns=[*keys, 'mae', 'rmse', 'mape_pct', 'n_points', 'n_mape'])
    rows = []
    for group_keys, chunk in df.groupby(keys, dropna=False, observed=True):
        if not isinstance(group_keys, tuple):
            group_keys = (group_keys,)
        yt = chunk.y_true.to_numpy(dtype=np.float64)
        yp = chunk.y_pred.to_numpy(dtype=np.float64)
        ae = np.abs(yp - yt)
        nz = np.abs(yt) > MAPE_EPS
        rows.append(dict(zip(keys, group_keys), mae=float(np.mean(ae)),
                         rmse=float(np.sqrt(np.mean(ae ** 2))),
                         mape_pct=float(np.mean(100 * ae[nz] / np.abs(yt[nz]))) if nz.any() else np.nan,
                         n_points=len(chunk), n_mape=int(nz.sum())))
    return pd.DataFrame(rows)


def summarize_fold_metrics(fold_metrics, group_keys):
    if fold_metrics.empty:
        return pd.DataFrame()
    return fold_metrics.groupby(group_keys, as_index=False, observed=True).agg(
        mae_mean=('mae', 'mean'), mae_std=('mae', 'std'),
        rmse_mean=('rmse', 'mean'), rmse_std=('rmse', 'std'),
        mape_mean_pct=('mape_pct', 'mean'), mape_std_pct=('mape_pct', 'std'),
        n_folds=('fold', 'nunique'), total_points=('n_points', 'sum'),
    )


def save_predictions_and_metrics(table, out, split):
    if table.empty:
        raise ValueError(f'No valid {split} masked prediction points')
    table.to_csv(out / f'cv5_{split}_predictions_pooled.csv', index=False)
    fold = add_metrics(table, ['fold', 'target', 'model'])
    fold.to_csv(out / f'cv5_{split}_fold_metrics.csv', index=False)
    summarize_fold_metrics(fold, ['target', 'model']).to_csv(
        out / f'cv5_{split}_summary_mean_std.csv', index=False)
    pooled = add_metrics(table, ['target', 'model'])
    pooled.to_csv(out / f'cv5_pooled_{split}_metrics.csv', index=False)
    gap_table = table.loc[table.gap_region.notna()]
    gap_table.to_csv(out / f'cv5_{split}_predictions_pooled_with_gap_region.csv', index=False)
    gap_fold = add_metrics(gap_table, ['fold', 'target', 'model', 'gap_region'])
    gap_fold.to_csv(out / f'cv5_{split}_fold_metrics_by_gap_region.csv', index=False)
    summarize_fold_metrics(gap_fold, ['target', 'model', 'gap_region']).to_csv(
        out / f'cv5_{split}_summary_mean_std_by_gap_region.csv', index=False)
    add_metrics(gap_table, ['target', 'model', 'gap_region']).to_csv(
        out / f'cv5_pooled_{split}_metrics_by_gap_region.csv', index=False)
    return fold


def main():
    if 'SAITSImputerStandard' not in globals() or 'saits_losses_standard' not in globals():
        raise NameError(
            'Run saits_official_architecture_1080x2.py first in the same namespace; '
            'the training script does not redefine the architecture. For a file use %run -i.'
        )
    seed_all(SEED)
    out = Path(OUT_DIR)
    (out / 'ckpts').mkdir(parents=True, exist_ok=True)
    with np.load(NPZ_PATH, allow_pickle=True) as data:
        keys = ['X_in', 'M_hat', 'X', 'I', 'segment_ids']
        missing = [k for k in keys if k not in data.files]
        if missing: raise KeyError(f'Missing NPZ keys: {missing}')
        x_in = np.asarray(data['X_in'], dtype=np.float32)
        m_hat = np.asarray(data['M_hat'], dtype=np.float32)
        x_true = np.asarray(data['X'], dtype=np.float32)
        i_mask = np.asarray(data['I'], dtype=np.float32)
        seg_id = np.asarray(data['segment_ids'])
        filenames = np.asarray(data['filenames']).astype(str) if 'filenames' in data.files else np.array([f'segment_{i}' for i in range(len(seg_id))])
        key = 'target_columns' if 'target_columns' in data.files and len(data['target_columns']) == D_IN else 'columns_all'
        target_names = np.asarray(data[key]).astype(str).tolist() if key in data.files and len(data[key]) == D_IN else ['ScO2', 'MAP']
    if not (x_in.shape == m_hat.shape == x_true.shape == i_mask.shape):
        raise ValueError('Shapes of X_in, M_hat, X, I differ')
    if x_in.ndim != 3 or x_in.shape[1:] != (SEQ_LEN, D_IN):
        raise ValueError(f'Expected (N,{SEQ_LEN},{D_IN}), got {x_in.shape}')
    if len(seg_id) != len(x_in):
        raise ValueError('segment_ids length mismatch')
    if not np.isin(m_hat, [0., 1.]).all() or not np.isin(i_mask, [0., 1.]).all():
        raise ValueError('Observed and artificial target masks must be binary')
    if np.any((m_hat > 0.5) & (i_mask > 0.5)):
        raise ValueError('M_hat and I overlap')
    if not np.isfinite(x_true[m_hat > 0.5]).all() or not np.isfinite(x_true[i_mask > 0.5]).all():
        raise ValueError('X has nonfinite observed or artificially withheld targets')
    if not np.isfinite(x_in[m_hat > 0.5]).all():
        raise ValueError('X_in has nonfinite observed inputs')
    if not np.allclose(x_in[m_hat > 0.5], x_true[m_hat > 0.5], rtol=1e-4, atol=1e-4):
        raise ValueError('X_in and X differ at observed input positions')

    pats = np.unique(seg_id)
    if len(pats) <= N_SPLITS:
        raise ValueError('Too few unique patients for five-fold train/val/test splitting')
    rng = np.random.default_rng(SEED)
    rng.shuffle(pats)
    folds = np.array_split(pats, N_SPLITS)
    folds_rows = []
    test_preds, val_preds, fold_results = [], [], []
    print('Device:', DEVICE, '| Data:', x_in.shape, '| Patients:', len(pats), '| Model:', MODEL_NAME)
    print('Channels:', target_names, '| AMP:', AMP_ENABLED, '| normalization:', NORMALIZE_WITH_TRAIN_STATS)
    print('SAITS train objective: ORT weight', LAM_ORT, '+ MIT weight', LAM_MIT)
    for fold, test_patients in enumerate(folds, 1):
        other = pats[~np.isin(pats, test_patients)].copy()
        vrng = np.random.default_rng(SEED + fold)
        vrng.shuffle(other)
        n_val = max(1, min(len(other) - 1, round(len(other) * VAL_WITHIN_TRAIN_P)))
        val_patients, train_patients = other[:n_val], other[n_val:]
        train_idx = np.flatnonzero(np.isin(seg_id, train_patients))
        val_idx = np.flatnonzero(np.isin(seg_id, val_patients))
        test_idx = np.flatnonzero(np.isin(seg_id, test_patients))
        if min(len(train_idx), len(val_idx), len(test_idx)) == 0:
            raise ValueError(f'Fold {fold}: empty split')
        folds_rows.append(dict(
            fold=fold,
            train_patients=','.join(map(str, train_patients)),
            val_patients=','.join(map(str, val_patients)),
            test_patients=','.join(map(str, test_patients)),
            n_train_patients=len(train_patients), n_val_patients=len(val_patients), n_test_patients=len(test_patients),
            n_train_segments=len(train_idx), n_val_segments=len(val_idx), n_test_segments=len(test_idx),
        ))
        means, stds = compute_train_stats(x_in, m_hat, train_idx)
        print(f'\nFOLD {fold}/{N_SPLITS} patients train/val/test={len(train_patients)}/{len(val_patients)}/{len(test_patients)} '
              f'| segments={len(train_idx)}/{len(val_idx)}/{len(test_idx)}', flush=True)
        print('  Train-only mean/std:', means, stds)
        loaders = {
            'train': load_loader(train_idx, True, x_in, m_hat, x_true, i_mask, means, stds),
            'val': load_loader(val_idx, False, x_in, m_hat, x_true, i_mask, means, stds, evaluation=True),
            'test': load_loader(test_idx, False, x_in, m_hat, x_true, i_mask, means, stds, evaluation=True),
        }
        seed_all(SEED + fold)
        model = SAITSImputerStandard(**MODEL_KWARGS).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        scaler = make_scaler()
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', factor=0.5, patience=3)
        checkpoint = out / 'ckpts' / f'fold{fold:02d}_best.pt'
        best, best_epoch, stale = float('inf'), -1, 0
        history = []
        for epoch in range(1, EPOCHS + 1):
            tr = run_loss_epoch(model, loaders['train'], optimizer=opt, scaler=scaler)
            va = run_loss_epoch(model, loaders['val'])
            if not np.isfinite(va['mit']) or va['n_mit'] == 0:
                raise ValueError(f'Fold {fold}: no finite held-out validation target points')
            scheduler.step(va['mit'])
            history.append(dict(epoch=epoch, lr=opt.param_groups[0]['lr'],
                                train_total_mae=tr['total'], train_ort_mae=tr['ort'], train_mit_mae=tr['mit'],
                                val_total_mae=va['total'], val_ort_mae=va['ort'], val_mit_mae=va['mit'],
                                n_val_mit=va['n_mit']))
            print(f'Epoch {epoch:03d} lr={opt.param_groups[0]["lr"]:.2e} | '
                  f'TRAIN total={tr["total"]:.5f} (ORT={tr["ort"]:.5f}, MIT={tr["mit"]:.5f}) | '
                  f'VAL total={va["total"]:.5f} (ORT={va["ort"]:.5f}, MIT={va["mit"]:.5f})', flush=True)
            if va['mit'] < best:
                best, best_epoch, stale = va['mit'], epoch, 0
                torch.save({'model_state_dict': model.state_dict(),
                            'model_kwargs': MODEL_KWARGS,
                            'channel_mean': means,
                            'channel_std': stds,
                            'best_val_mit_mae_normalized': best,
                            'fold': fold, 'epoch': epoch}, checkpoint)
                print('  Saved best checkpoint:', checkpoint, flush=True)
            else:
                stale += 1
                if stale >= EARLY_STOPPING_PATIENCE:
                    print('  Early stopping:', epoch, flush=True)
                    break
        pd.DataFrame(history).to_csv(out / f'fold{fold:02d}_training_history.csv', index=False)
        state = torch.load(checkpoint, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state['model_state_dict'])
        te = run_loss_epoch(model, loaders['test'])
        va = run_loss_epoch(model, loaders['val'])
        pred_val = collect_predictions(model, loaders['val'], fold, 'val', seg_id, filenames, target_names, means, stds)
        pred_test = collect_predictions(model, loaders['test'], fold, 'test', seg_id, filenames, target_names, means, stds)
        if pred_test.empty or pred_val.empty:
            raise ValueError(f'Fold {fold}: empty validation or test masked predictions')
        pred_val.to_csv(out / f'fold{fold:02d}_validation_predictions.csv', index=False)
        pred_test.to_csv(out / f'fold{fold:02d}_test_predictions.csv', index=False)
        val_preds.append(pred_val)
        test_preds.append(pred_test)
        fold_results.append(dict(fold=fold, best_epoch=best_epoch,
                                 val_mit_mae_normalized=best,
                                 test_mit_mae_normalized=te['mit'],
                                 test_total_mae_normalized=te['total'],
                                 val_total_mae_normalized=va['total']))
        for channel, group in pred_test.groupby('target'):
            m = add_metrics(group, ['target']).iloc[0]
            print(f'  {channel}: TEST MAE={m.mae:.5f}, RMSE={m.rmse:.5f}, '
                  f'MAPE={m.mape_pct:.3f}%, n={int(m.n_points)}', flush=True)
        del model
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    pd.DataFrame(folds_rows).to_csv(out / 'cv5_patient_split_summary.csv', index=False)
    test = pd.concat(test_preds, ignore_index=True)
    val = pd.concat(val_preds, ignore_index=True)
    val_fold = save_predictions_and_metrics(val, out, 'validation')
    test_fold = save_predictions_and_metrics(test, out, 'test')
    combined = test_fold.merge(pd.DataFrame(fold_results), on='fold', how='left')
    combined.to_csv(out / 'cv5_fold_results.csv', index=False)
    summarize_fold_metrics(combined, ['target', 'model']).to_csv(out / 'cv5_summary_mean_std.csv', index=False)
    # legacy aliases used by downstream reporting scripts
    add_metrics(test, ['target', 'model']).to_csv(out / 'cv5_pooled_test_metrics.csv', index=False)
    add_metrics(test.loc[test.gap_region.notna()], ['target','model','gap_region']).to_csv(
        out / 'cv5_pooled_test_metrics_by_gap_region.csv', index=False)
    print('\nFive-fold test summary:\n',
          summarize_fold_metrics(combined, ['target', 'model']).to_string(index=False))
    print('\nSaved checkpoints, histories, splits, validation/test predictions and summaries to:', out)

if __name__ == '__main__':
    main()

Device: cuda | Data: (3817, 1080, 2) | Patients: 64 | Model: SAITS_OfficialArchitecture
Channels: ['ScO2', 'MAP'] | AMP: True | normalization: True
SAITS train objective: ORT weight 1.0 + MIT weight 1.0

FOLD 1/5 patients train/val/test=46/5/13 | segments=2657/413/747
  Train-only mean/std: [61.97329 58.27655] [12.682247 13.777719]
Epoch 001 lr=1.00e-03 | TRAIN total=2.66162 (ORT=1.39120, MIT=1.27042) | VAL total=1.58371 (ORT=0.94871, MIT=0.63500)
  Saved best checkpoint: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64/ckpts/fold01_best.pt
Epoch 002 lr=1.00e-03 | TRAIN total=1.03268 (ORT=0.63663, MIT=0.39605) | VAL total=0.77254 (ORT=0.50031, MIT=0.27224)
  Saved best checkpoint: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64/ckpts/fold01_best.pt
Epoch 003 lr=1.00e-03 | TRAIN total=0.67005 (ORT=0.38577, MIT=0.28428) | VAL total=0.71705 (ORT=0.36440, MIT=0.35265)
Epoch 004 lr=1.00e-03 | TRAIN total=0.62272 (ORT=0.33066, MIT=0.29206) | VAL total

### Validation loss

In [6]:
"""Re-evaluate 5-fold SAITS validation from existing saved checkpoints. NO training.

Run the SAITS model-definition cell/file first, in the SAME Colab namespace:
  %run -i /content/drive/MyDrive/aim3/saits_official_architecture_1080x2.py
  %run -i /content/drive/MyDrive/aim3/recompute_saits_official_5fold_validation.py

This file does NOT define SAITSImputerStandard or saits_losses_standard.
It is paired with train_saits_official_architecture_5fold.py and its checkpoint
format: each checkpoint stores model_state_dict, model_kwargs, channel_mean,
channel_std, fold, and best_val_mit_mae_normalized. The model receives ONLY
X_in and M_hat, never artificially hidden ground-truth values.

Two distinct outputs:
  - ORT, MIT, combined SAITS MAE objectives in TRAIN-standardized units.
  - Held-out validation imputation MAE/RMSE/MAPE in PHYSICAL units, per channel,
    per gap duration, fold means +/- sample standard deviations and pooled.

Recomputed outputs are suffixed _recomputed so training CSVs are not overwritten.
"""

import os
import random
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# ===================== CONFIG (match training) =====================
SPLIT_CSV = os.path.join(OUT_DIR, 'cv5_patient_split_summary.csv')
CKPT_DIR = os.path.join(OUT_DIR, 'ckpts')
MODEL_NAME = 'SAITS_OfficialArchitecture'
SEQ_LEN, D_IN = 1080, 2
BATCH_SIZE = 4            
NUM_WORKERS = 0
SEED = 1337
LAM_ORT, LAM_MIT = 1.0, 1.0  
USE_AMP = True         
MAPE_EPS = 1e-8
COLLECT_POINTWISE_PREDICTIONS = True
REGION_ORDER = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PIN_MEMORY = DEVICE.type == 'cuda'
AMP_ENABLED = USE_AMP and DEVICE.type == 'cuda'
AMP_DTYPE = (torch.bfloat16 if AMP_ENABLED and torch.cuda.is_bf16_supported()
             else torch.float16)


def autocast_ctx():
    return torch.autocast('cuda', dtype=AMP_DTYPE) if AMP_ENABLED else nullcontext()


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def parse_patients(value):
    if pd.isna(value) or not str(value).strip():
        return []
    return [int(float(s.strip())) for s in str(value).split(',') if s.strip()]


class ValidationDataset(Dataset):
    """Replicates TRAINING script's observed-mask/train-stat normalization."""
    def __init__(self, x_in, m_hat, truth, i_mask, indices, means, stds):
        mu = np.asarray(means, dtype=np.float32).reshape(1, 1, -1)
        sd = np.asarray(stds, dtype=np.float32).reshape(1, 1, -1)
        visible = np.asarray(m_hat) > 0.5
        xin = np.where(visible, (np.asarray(x_in, np.float32)-mu)/sd, 0.)
        truth = np.asarray(truth, np.float32)
        gt = np.where(np.isfinite(truth), (truth-mu)/sd, np.nan)
        self.x = torch.from_numpy(np.nan_to_num(xin, nan=0., posinf=0., neginf=0.).astype(np.float32))
        self.m = torch.from_numpy(visible.astype(np.float32))
        self.y = torch.from_numpy(gt.astype(np.float32))
        self.i = torch.from_numpy((np.asarray(i_mask)>0.5).astype(np.float32))
        self.indices = torch.as_tensor(np.asarray(indices, dtype=np.int64))
        if not (self.x.shape == self.m.shape == self.y.shape == self.i.shape):
            raise ValueError('Validation X_in/M_hat/X/I shapes do not match')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.x[i], self.m[i], self.y[i], self.i[i], self.indices[i]


def gap_lengths(mask):
    """Full contiguous hidden-run length assigned to each hidden time point."""
    a = np.asarray(mask) > 0.5
    bounds = np.diff(np.r_[False, a, False].astype(np.int8))
    starts = np.flatnonzero(bounds == 1)
    stops = np.flatnonzero(bounds == -1)
    lengths = np.zeros(len(a), dtype=np.int32)
    for start, stop in zip(starts, stops):
        lengths[start:stop] = stop-start
    return lengths


def gap_region(g):
    if 1 <= g < 12: return '<1 min'
    if 12 <= g < 120: return '[1,10) min'
    if 120 <= g < 240: return '[10,20) min'
    if 240 <= g <= 360: return '[20,30] min'
    return None


def calc_metrics(df):
    if df.empty:
        return dict(mae=np.nan, rmse=np.nan, mape_pct=np.nan, n_points=0, n_mape=0)
    yt = df.y_true.to_numpy(dtype=np.float64)
    yp = df.y_pred.to_numpy(dtype=np.float64)
    good = np.isfinite(yt) & np.isfinite(yp)
    yt, yp = yt[good], yp[good]
    if len(yt) == 0:
        return dict(mae=np.nan, rmse=np.nan, mape_pct=np.nan, n_points=0, n_mape=0)
    ae = np.abs(yp-yt)
    nz = np.abs(yt) > MAPE_EPS
    return dict(mae=float(ae.mean()), rmse=float(np.sqrt((ae*ae).mean())),
                mape_pct=float((100*ae[nz]/np.abs(yt[nz])).mean()) if nz.any() else np.nan,
                n_points=int(len(yt)), n_mape=int(nz.sum()))


def grouped_metrics(df, columns):
    if df.empty:
        return pd.DataFrame(columns=[*columns, 'mae', 'rmse', 'mape_pct', 'n_points', 'n_mape'])
    records = []
    for key, block in df.groupby(columns, observed=True, dropna=False):
        if not isinstance(key, tuple):
            key = (key,)
        records.append(dict(zip(columns, key), **calc_metrics(block)))
    return pd.DataFrame(records)


def summary_metrics(fold_metrics, keys):
    if fold_metrics.empty:
        return pd.DataFrame()
    return fold_metrics.groupby(keys, observed=True, as_index=False).agg(
        mae_mean=('mae', 'mean'), mae_std=('mae', 'std'),
        rmse_mean=('rmse', 'mean'), rmse_std=('rmse', 'std'),
        mape_mean_pct=('mape_pct', 'mean'), mape_std_pct=('mape_pct', 'std'),
        n_folds=('fold', 'nunique'), total_points=('n_points', 'sum'))


def save(df, filename):
    p = Path(OUT_DIR) / filename
    df.to_csv(p, index=False)
    print('Saved:', p)
    return p


def main():
    if 'SAITSImputerStandard' not in globals() or 'saits_losses_standard' not in globals():
        raise NameError('Run saits_official_architecture_1080x2.py first in the same '
                        'namespace (use %run -i). This script does NOT redefine SAITS.')
    if not os.path.isfile(SPLIT_CSV):
        raise FileNotFoundError('Missing saved training split CSV: '+SPLIT_CSV)
    seed_all(SEED)
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

    with np.load(NPZ_PATH, allow_pickle=True) as d:
        keys = ['X_in', 'M_hat', 'X', 'I', 'segment_ids']
        missing = [k for k in keys if k not in d.files]
        if missing: raise KeyError('Missing NPZ arrays: '+str(missing))
        x_in, m_hat, truth, i_mask = [np.asarray(d[k], dtype=np.float32) for k in keys[:4]]
        seg_id = np.asarray(d['segment_ids'])
        filenames = (np.asarray(d['filenames']).astype(str) if 'filenames' in d.files
                     else np.array([f'segment_{i}' for i in range(len(seg_id))]))
        if 'target_columns' in d.files and len(d['target_columns']) == D_IN:
            target_names = np.asarray(d['target_columns']).astype(str).tolist()
        elif 'columns_all' in d.files and len(d['columns_all']) == D_IN:
            target_names = np.asarray(d['columns_all']).astype(str).tolist()
        else:
            target_names = ['ScO2', 'MAP']

    if not (x_in.shape == m_hat.shape == truth.shape == i_mask.shape):
        raise ValueError('NPZ X_in/M_hat/X/I shapes differ')
    if x_in.ndim != 3 or x_in.shape[1:] != (SEQ_LEN, D_IN):
        raise ValueError(f'Expected (N,{SEQ_LEN},{D_IN}), found {x_in.shape}')
    if len(seg_id) != len(x_in):
        raise ValueError('segment_ids length differs from N')
    if not np.isin(m_hat, [0., 1.]).all() or not np.isin(i_mask, [0., 1.]).all():
        raise ValueError('M_hat and I must be binary')
    if np.any((m_hat > 0.5) & (i_mask > 0.5)):
        raise ValueError('M_hat and I overlap; withheld points cannot be observed inputs')
    if not np.isfinite(truth[(i_mask>0.5)|(m_hat>0.5)]).all():
        raise ValueError('X contains nonfinite observed or artificial target values')
    if not np.isfinite(x_in[m_hat>0.5]).all():
        raise ValueError('Observed input is nonfinite')
    if not np.allclose(x_in[m_hat>0.5], truth[m_hat>0.5], rtol=1e-4, atol=1e-4):
        raise ValueError('Observed X_in differs from original observed X')
    splits = pd.read_csv(SPLIT_CSV).sort_values('fold')
    if len(splits) != 5 or splits.fold.nunique() != 5:
        raise ValueError('Expected five saved patient-fold rows')
    if len(set(splits.fold.astype(int))) != 5:
        raise ValueError('Duplicate fold IDs')
    print('Device:', DEVICE, 'AMP:', AMP_ENABLED, '| NPZ:', x_in.shape)
    print('Output:', OUT_DIR, '| Targets:', target_names)
    print('Validation SAITS objective: (ORT x', LAM_ORT, ')+ (MIT x', LAM_MIT, '), normalized units')

    losses, predictions = [], []
    for _, split in splits.iterrows():
        fold = int(split.fold)
        val_patients = parse_patients(split.val_patients)
        indices = np.flatnonzero(np.isin(seg_id, val_patients))
        if len(indices) == 0:
            raise ValueError(f'Fold {fold}: no validation segments')
        if 'n_val_segments' in split and not pd.isna(split.n_val_segments):
            if len(indices) != int(split.n_val_segments):
                raise ValueError(f'Fold {fold}: validation indices mismatch training splits')
        ckpt_file = Path(CKPT_DIR) / f'fold{fold:02d}_best.pt'
        if not ckpt_file.is_file():
            raise FileNotFoundError(str(ckpt_file))
        # Trusted own checkpoints contain NumPy arrays of channel statistics;
        # they may require weights_only=False on recent PyTorch versions.
        checkpoint = torch.load(ckpt_file, map_location='cpu', weights_only=False)
        required = {'model_state_dict', 'model_kwargs', 'channel_mean', 'channel_std'}
        missing_ckpt = required - set(checkpoint)
        if missing_ckpt:
            raise ValueError(f'Fold {fold}: checkpoint lacks {missing_ckpt}; expected new SAITS training format')
        model_kwargs = dict(checkpoint['model_kwargs'])
        if model_kwargs.get('seq_len') != SEQ_LEN or model_kwargs.get('d_in') != D_IN:
            raise ValueError(f'Fold {fold}: checkpoint model input dimensions differ')
        mu = np.asarray(checkpoint['channel_mean'], dtype=np.float32)
        sd = np.asarray(checkpoint['channel_std'], dtype=np.float32)
        if mu.shape != (D_IN,) or sd.shape != (D_IN,) or not np.isfinite(mu).all() or not np.isfinite(sd).all() or (sd<=0).any():
            raise ValueError(f'Fold {fold}: invalid checkpoint normalization statistics')
        loader = DataLoader(
            ValidationDataset(x_in[indices], m_hat[indices], truth[indices], i_mask[indices],
                              indices, mu, sd),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
            pin_memory=PIN_MEMORY)
        model = SAITSImputerStandard(**model_kwargs).to(DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'], strict=True)
        model.eval()

        ort_num = mit_num = 0.0
        ort_n = mit_n = 0
        rows = []
        with torch.inference_mode():
            for xb, mb, yb, ib, idxs in loader:
                xb = xb.to(DEVICE, non_blocking=True)
                mb = mb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                ib = ib.to(DEVICE, non_blocking=True)
                n_ort = int(((mb>0.5) & torch.isfinite(yb)).sum().item())
                n_mit = int(((ib>0.5) & (mb<=0.5) & torch.isfinite(yb)).sum().item())
                if not n_ort and not n_mit:
                    continue
                with autocast_ctx():
                    t1, t2, t3, completed, _ = model(xb, mb)
                    _, ort, mit = saits_losses_standard(
                        t1, t2, t3, completed, yb, mb, ib,
                        lam_ort=LAM_ORT, lam_mit=LAM_MIT)
                if not (torch.isfinite(ort).item() and torch.isfinite(mit).item()):
                    raise FloatingPointError(f'Fold {fold}: nonfinite validation SAITS losses')
                ort_num += float(ort.item()) * n_ort
                mit_num += float(mit.item()) * n_mit
                ort_n += n_ort
                mit_n += n_mit

                if COLLECT_POINTWISE_PREDICTIONS:
                    pred_np = completed.float().cpu().numpy() * sd[None,None,:] + mu[None,None,:]
                    raw_gt = yb.cpu().numpy() * sd[None,None,:] + mu[None,None,:]
                    im_np = ib.cpu().numpy()
                    for b, idx in enumerate(idxs.tolist()):
                        for d, target in enumerate(target_names):
                            lengths = gap_lengths(im_np[b, :, d])
                            for t in np.flatnonzero(im_np[b, :, d]>0.5):
                                yt = float(raw_gt[b,t,d]); yp = float(pred_np[b,t,d])
                                if not (np.isfinite(yt) and np.isfinite(yp)):
                                    continue
                                g = int(lengths[t])
                                rows.append(dict(fold=fold, split='val', model=MODEL_NAME,
                                    segment_idx=int(idx), patient_id=str(seg_id[idx]),
                                    filename=str(filenames[idx]), time_idx=int(t),
                                    target=str(target), y_true=yt, y_pred=yp,
                                    abs_err=abs(yt-yp), gap_len=g, gap_region=gap_region(g)))

        if mit_n == 0:
            raise ValueError(f'Fold {fold}: zero eligible masked validation points')
        va_ort = ort_num/ort_n if ort_n else np.nan
        va_mit = mit_num/mit_n
        va_total = LAM_ORT*(va_ort if np.isfinite(va_ort) else 0.0)+LAM_MIT*va_mit
        saved_best = float(checkpoint.get('best_val_mit_mae_normalized', np.nan))
        losses.append(dict(fold=fold, model=MODEL_NAME,
                           val_total_mae_normalized=va_total,
                           val_ort_mae_normalized=va_ort,
                           val_mit_mae_normalized=va_mit,
                           saved_best_val_mit_mae_normalized=saved_best,
                           recompute_minus_saved=va_mit-saved_best,
                           n_val_patients=len(val_patients), n_val_segments=len(indices),
                           n_ort_points=ort_n, n_mit_points=mit_n))
        print(f'FOLD {fold}/5: {len(val_patients)} validation patients, {len(indices)} segments, '
              f'{mit_n:,} artificial target points')
        print(f'  Normalized ORT MAE={va_ort:.6f}, MIT MAE={va_mit:.6f}, total={va_total:.6f}; '
              f'saved best MIT={saved_best:.6f}')
        if COLLECT_POINTWISE_PREDICTIONS:
            df = pd.DataFrame(rows)
            if df.empty:
                raise ValueError(f'Fold {fold}: no finite validation predictions')
            save(df, f'fold{fold:02d}_validation_predictions_recomputed.csv')
            predictions.append(df)
            for target, grp in df.groupby('target', observed=True):
                m = calc_metrics(grp)
                print(f'  {target} MAE={m["mae"]:.5f}, RMSE={m["rmse"]:.5f}, MAPE={m["mape_pct"]:.3f}%')
        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

    loss_df = pd.DataFrame(losses)
    save(loss_df, 'cv5_validation_losses_recomputed.csv')
    loss_summary = pd.DataFrame([dict(
        model=MODEL_NAME,
        val_total_mae_mean=loss_df.val_total_mae_normalized.mean(),
        val_total_mae_std=loss_df.val_total_mae_normalized.std(ddof=1),
        val_ort_mae_mean=loss_df.val_ort_mae_normalized.mean(),
        val_ort_mae_std=loss_df.val_ort_mae_normalized.std(ddof=1),
        val_mit_mae_mean=loss_df.val_mit_mae_normalized.mean(),
        val_mit_mae_std=loss_df.val_mit_mae_normalized.std(ddof=1),
        n_folds=len(loss_df))])
    save(loss_summary, 'cv5_validation_loss_summary_mean_std_recomputed.csv')
    print('\nFive-fold SAITS validation loss (normalized units):\n', loss_df.to_string(index=False))
    if not COLLECT_POINTWISE_PREDICTIONS:
        return

    pred_df = pd.concat(predictions, ignore_index=True)
    save(pred_df, 'cv5_validation_predictions_recomputed_pooled.csv')
    gap_pred = pred_df.loc[pred_df.gap_region.notna()].copy()
    save(gap_pred, 'cv5_validation_predictions_recomputed_with_gap_region.csv')
    fold_metrics = grouped_metrics(pred_df, ['fold','target','model'])
    save(fold_metrics, 'cv5_validation_fold_metrics_recomputed.csv')
    save(summary_metrics(fold_metrics, ['target','model']),
         'cv5_validation_metrics_recomputed_mean_std.csv')
    save(grouped_metrics(pred_df, ['target','model']),
         'cv5_validation_pooled_metrics_recomputed.csv')
    fold_gap = grouped_metrics(gap_pred, ['fold','target','model','gap_region'])
    save(fold_gap, 'cv5_validation_fold_metrics_recomputed_by_gap_region.csv')
    gap_summary = summary_metrics(fold_gap, ['target','model','gap_region'])
    save(gap_summary, 'cv5_validation_metrics_recomputed_mean_std_by_gap_region.csv')
    save(grouped_metrics(gap_pred, ['target','model','gap_region']),
         'cv5_validation_pooled_metrics_recomputed_by_gap_region.csv')
    print('\nVALIDATION IMPUTATION MAE/RMSE/MAPE, five-fold mean +/- SD (raw units):\n',
          summary_metrics(fold_metrics, ['target','model']).to_string(index=False))
    print('\nSAITS validation results saved in:', OUT_DIR)
    print("\n=== FIVE-FOLD OVERALL VALIDATION LOSS ===")

    print(
        f"MIT MAE: "
        f"{loss_df['val_mit_mae_normalized'].mean():.6f} ± "
        f"{loss_df['val_mit_mae_normalized'].std(ddof=1):.6f}"
    )

    print(
        f"Total MAE: "
        f"{loss_df['val_total_mae_normalized'].mean():.6f} ± "
        f"{loss_df['val_total_mae_normalized'].std(ddof=1):.6f}"
    )


if __name__ == '__main__':
    main()

Device: cuda AMP: True | NPZ: (3817, 1080, 2)
Output: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64 | Targets: ['ScO2', 'MAP']
Validation SAITS objective: (ORT x 1.0 )+ (MIT x 1.0 ), normalized units
FOLD 1/5: 5 validation patients, 413 segments, 14,265 artificial target points
  Normalized ORT MAE=0.019202, MIT MAE=0.109029, total=0.128231; saved best MIT=0.109029
Saved: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64/fold01_validation_predictions_recomputed.csv
  MAP MAE=2.13441, RMSE=4.32596, MAPE=3.126%
  ScO2 MAE=1.18365, RMSE=2.17934, MAPE=2.005%
FOLD 2/5: 5 validation patients, 293 segments, 11,136 artificial target points
  Normalized ORT MAE=0.296070, MIT MAE=0.094326, total=0.390396; saved best MIT=0.094326
Saved: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64/fold02_validation_predictions_recomputed.csv
  MAP MAE=1.59532, RMSE=3.02605, MAPE=3.108%
  ScO2 MAE=1.16679, RMSE=1.83178, MAPE=1.992%
FOLD 3/5: 5 valid

### Model testing

In [7]:
import os
import numpy as np
import pandas as pd
# ============================ CONFIG ============================

PRED_CSV = os.path.join(OUT_DIR, 'cv5_test_predictions_pooled.csv')
MODEL_NAME = 'SAITS_OfficialArchitecture'
EPS = 1e-8
N_FOLDS = 5
REGIONS = ['<1 min', '[1,10) min', '[10,20) min', '[20,30] min', 'ALL']
PREFIX = 'saits_paper4'


def calculate_metrics(frame):
    """Absolute MAE; percentage MAPE excluding only near-zero ground truth."""
    yt = frame['y_true'].to_numpy(dtype=np.float64)
    yp = frame['y_pred'].to_numpy(dtype=np.float64)
    ae = np.abs(yp - yt)
    valid_mape = np.abs(yt) > EPS
    return {
        'MAE': float(ae.mean()) if len(ae) else np.nan,
        'MAPE_pct': (
            float(100.0 * np.mean(ae[valid_mape] / np.abs(yt[valid_mape])))
            if valid_mape.any() else np.nan
        ),
        'n_points': int(len(yt)),
        'n_mape_points': int(valid_mape.sum()),
        'n_excluded_gt0': int((~valid_mape).sum()),
    }


def group_metrics(frame, group_cols):
    records = []
    for group_key, sub in frame.groupby(group_cols, observed=True, sort=False):
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        records.append({**dict(zip(group_cols, group_key)), **calculate_metrics(sub)})
    return pd.DataFrame(records)


def fmt(mean, sd):
    if pd.isna(mean):
        return 'NA'
    if pd.isna(sd):
        return f'{mean:.2f} ± NA'
    return f'{mean:.2f} ± {sd:.2f}'


def main():
    if not os.path.isfile(PRED_CSV):
        raise FileNotFoundError(
            f'Missing test predictions: {PRED_CSV}\n'
            'Run train_saits_official_architecture_5fold.py first.'
        )
    pred = pd.read_csv(PRED_CSV)
    required = {'fold', 'target', 'y_true', 'y_pred', 'gap_len'}
    missing = required - set(pred.columns)
    if missing:
        raise ValueError(f'Missing required columns: {sorted(missing)}')

    # Prevent accidentally including validation or other models.
    if 'split' in pred.columns:
        pred = pred.loc[pred['split'].astype(str).str.lower() == 'test'].copy()
    if 'model' not in pred.columns:
        pred['model'] = MODEL_NAME
    else:
        found_models = sorted(pred['model'].dropna().astype(str).unique().tolist())
        if found_models != [MODEL_NAME]:
            raise ValueError(
                f'Expected only {MODEL_NAME!r}, but found models {found_models}. '
                'Check OUT_DIR and ensure this is the new SAITS training output.'
            )

    for name in ['fold', 'gap_len', 'y_true', 'y_pred']:
        pred[name] = pd.to_numeric(pred[name], errors='coerce')
    pred = pred.loc[
        np.isfinite(pred['fold']) & np.isfinite(pred['y_true'])
        & np.isfinite(pred['y_pred'])
    ].copy()
    if pred.empty:
        raise ValueError('No valid held-out TEST predictions available.')
    if not np.all(pred['fold'] == pred['fold'].astype(int)):
        raise ValueError('Fold identifiers must be integers.')
    pred['fold'] = pred['fold'].astype(int)
    actual_folds = sorted(pred['fold'].unique().tolist())
    if actual_folds != list(range(1, N_FOLDS + 1)):
        raise ValueError(f'Expected folds 1..{N_FOLDS}, found {actual_folds}.')

    pred['gap_region'] = pd.cut(
        pred['gap_len'], bins=[0, 11, 119, 239, 360],
        labels=REGIONS[:-1], include_lowest=True,
    )
    outside = int(pred['gap_region'].isna().sum())
    if outside:
        print(f'WARNING: {outside} valid test points have gap length outside 1..360 '
              'or missing; they remain in ALL but not the four length groups.')

    print('Channel labels in saved predictions:', sorted(pred['target'].unique().tolist()))
    print('Test points by gap duration:\n', pred['gap_region'].value_counts(sort=False).to_string())
    print('ALL eligible test points:', len(pred))

    # Extra "ALL" rows recompute actual pointwise metrics (not a mean of bin means).
    specific = pred.loc[pred['gap_region'].notna()].copy()
    all_points = pred.copy()
    all_points['gap_region'] = 'ALL'
    combined = pd.concat([specific, all_points], ignore_index=True)
    combined['gap_region'] = pd.Categorical(
        combined['gap_region'].astype(str), categories=REGIONS, ordered=True,
    )

    foldwise = group_metrics(combined, ['fold', 'target', 'model', 'gap_region'])
    foldwise['gap_region'] = pd.Categorical(
        foldwise['gap_region'], categories=REGIONS, ordered=True,
    )
    foldwise = foldwise.sort_values(
        ['target', 'model', 'gap_region', 'fold']
    ).reset_index(drop=True)

    summary = (
        foldwise.groupby(['target', 'model', 'gap_region'], observed=True)
        .agg(
            MAE_mean=('MAE', 'mean'),
            MAE_std=('MAE', 'std'),          
            MAPE_mean_pct=('MAPE_pct', 'mean'),
            MAPE_std_pct=('MAPE_pct', 'std'),
            n_folds_MAE=('MAE', 'count'),
            n_folds_MAPE=('MAPE_pct', 'count'),
            total_points=('n_points', 'sum'),
            total_mape_points=('n_mape_points', 'sum'),
            total_excluded_gt0=('n_excluded_gt0', 'sum'),
        )
        .reset_index()
        .sort_values(['target', 'model', 'gap_region'])
        .reset_index(drop=True)
    )

    pooled = group_metrics(combined, ['target', 'model', 'gap_region'])
    pooled['gap_region'] = pd.Categorical(
        pooled['gap_region'], categories=REGIONS, ordered=True,
    )
    pooled = pooled.sort_values(
        ['target', 'model', 'gap_region']
    ).reset_index(drop=True)

    paper_rows = []
    for (target, model), sub in summary.groupby(['target', 'model']):
        row = {'target': target, 'model': model}
        for region in REGIONS:
            reg = sub.loc[sub['gap_region'] == region]
            row[f'{region} | MAE'] = (
                fmt(reg.iloc[0]['MAE_mean'], reg.iloc[0]['MAE_std'])
                if not reg.empty else 'NA'
            )
            row[f'{region} | MAPE(%)'] = (
                fmt(reg.iloc[0]['MAPE_mean_pct'], reg.iloc[0]['MAPE_std_pct'])
                if not reg.empty else 'NA'
            )
        paper_rows.append(row)
    paper = pd.DataFrame(paper_rows)

    os.makedirs(OUT_DIR, exist_ok=True)
    outputs = {
        f'{PREFIX}_foldwise.csv': foldwise,
        f'{PREFIX}_mean_std.csv': summary,
        f'{PREFIX}_pooled.csv': pooled,
        f'{PREFIX}_paper_table.csv': paper,
    }
    for filename, table in outputs.items():
        path = os.path.join(OUT_DIR, filename)
        table.to_csv(path, index=False)
        print('Saved:', path)

    print('\nSAITS FIVE-FOLD TEST MAE/MAPE:')
    print(summary.to_string(index=False))
    print('\nSAITS MANUSCRIPT-READY TABLE:')
    print(paper.to_string(index=False))
    return foldwise, summary, pooled, paper


if __name__ == '__main__':
    main()


Channel labels in saved predictions: ['MAP', 'ScO2']
Test points by gap duration:
 gap_region
<1 min         39503
[1,10) min     52673
[10,20) min    26458
[20,30] min    25794
ALL eligible test points: 144428
Saved: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64/saits_paper4_foldwise.csv
Saved: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64/saits_paper4_mean_std.csv
Saved: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64/saits_paper4_pooled.csv
Saved: /content/drive/MyDrive/aim3/SAITS_OfficialArchitecture_5fold_1e-3_64/saits_paper4_paper_table.csv

SAITS FIVE-FOLD TEST MAE/MAPE:
target                      model  gap_region  MAE_mean  MAE_std  MAPE_mean_pct  MAPE_std_pct  n_folds_MAE  n_folds_MAPE  total_points  total_mape_points  total_excluded_gt0
   MAP SAITS_OfficialArchitecture      <1 min  1.795634 0.327005       3.337203      1.176652            5             5         13967              13967                   0
